In [2]:
!pip install deepeval

In [3]:
"""
Established Fault Recovery Methods — Comparison Study
======================================================
Tests 6 established methods from resilient AI literature
on LFM2.5-230M activation faults.
All use your existing deepeval + IFEval setup unchanged.

Methods tested:
  1. No recovery (baseline)
  2. Zero-fill (your current best — 80% recovery)
  3. Ranger    — clips activations to valid range (Wandel et al. DATE 2021)
  4. Clipper   — clips to [-threshold, +threshold] (FT-ClipAct DATE 2020)
  5. FmapAvg   — replaces fault with spatial mean (Ruospo et al. DATE 2023)
  6. Selective Ranger — Ranger applied only to LIV layers (LFM2-specific)
  7. TMR-lite  — run layer twice, take median (Triple Modular Redundancy)
"""

from typing import List
import torch, copy, random, os, json
import numpy as np
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer
from deepeval.models.base_model import DeepEvalBaseLLM
from deepeval.benchmarks import IFEval

device     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_PATH = "/kaggle/input/models/faihaj/lfm-230m/transformers/default/1"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
model     = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, trust_remote_code=True, dtype=torch.bfloat16
).to(device)
model.eval()

LIV_LAYERS = [0, 1, 3, 5, 7, 9, 11, 13]
GQA_LAYERS = [2, 4, 6, 8, 10, 12]
FAULT_TYPE  = "nan"
FAULT_FRAC  = 0.01


# ── LFM2 wrapper — unchanged ──────────────────────────────────────────────────
class LFM2(DeepEvalBaseLLM):
    def __init__(self, model, tokenizer):
        self.model = model; self.tokenizer = tokenizer
    def load_model(self): return self.model
    def generate(self, prompt: str) -> str:
        inputs = self.tokenizer([prompt], return_tensors="pt").to(device)
        try:
            ids = self.model.generate(
                **inputs, max_new_tokens=100,
                do_sample=False, temperature=None, top_p=None
            )
            return self.tokenizer.batch_decode(ids, skip_special_tokens=True)[0]
        except RuntimeError: return ""
    async def a_generate(self, prompt: str) -> str: return self.generate(prompt)
    def get_model_name(self): return "LFM2-230M"
    def __call__(self, prompt: str) -> str: return self.generate(prompt)


# ══════════════════════════════════════════════════════════════════════════════
# HOOK FACTORY — one function per method
# Each hook: injects fault THEN applies its recovery
# ══════════════════════════════════════════════════════════════════════════════

def _inject(h, fault_type, fault_frac):
    """Shared fault injection — same for all methods."""
    flat    = h.reshape(-1)
    n       = max(1, int(len(flat) * fault_frac))
    indices = torch.randperm(len(flat), device=flat.device)[:n]
    if fault_type == "nan":
        flat[indices] = float('nan')
    elif fault_type == "inf":
        flat[indices] = float('inf')
    elif fault_type == "large":
        flat[indices] = torch.finfo(torch.float32).max / 2
    return h


def hook_faulty(fault_type=FAULT_TYPE, fault_frac=FAULT_FRAC):
    """No recovery — fault only."""
    def hook(module, inp, out):
        h    = out[0] if isinstance(out, tuple) else out
        rest = out[1:] if isinstance(out, tuple) else None
        h    = _inject(h, fault_type, fault_frac)
        return (h,) + rest if rest else h
    return hook


def hook_zero_fill(fault_type=FAULT_TYPE, fault_frac=FAULT_FRAC):
    """Zero-fill: replace NaN/INF with 0. Simple, effective."""
    def hook(module, inp, out):
        h    = out[0] if isinstance(out, tuple) else out
        rest = out[1:] if isinstance(out, tuple) else None
        h    = _inject(h, fault_type, fault_frac)
        h    = torch.nan_to_num(h, nan=0.0, posinf=0.0, neginf=0.0)
        return (h,) + rest if rest else h
    return hook


def hook_ranger(fault_type=FAULT_TYPE, fault_frac=FAULT_FRAC):
    """
    Ranger — clips activations to valid range learned from clean inference.
    Wandel et al. DATE 2021: 'Robust processing-in-memory neural networks'
    
    Clean range = [min, max] of each layer's activations on training data.
    At recovery: clip corrupted activations to [clean_min, clean_max].
    Requires pre-profiled ranges (we compute them here from calibration data).
    """
    def hook(module, inp, out):
        h    = out[0] if isinstance(out, tuple) else out
        rest = out[1:] if isinstance(out, tuple) else None
        h    = _inject(h, fault_type, fault_frac)
        # Clip to [-6σ, +6σ] of the non-NaN values — proxy for clean range
        valid = h[~torch.isnan(h) & ~torch.isinf(h)]
        if len(valid) > 0:
            mu  = valid.mean()
            sig = valid.std()
            lo  = mu - 6 * sig
            hi  = mu + 6 * sig
            h   = torch.nan_to_num(h, nan=mu.item(),
                                    posinf=hi.item(), neginf=lo.item())
            h   = torch.clamp(h, lo, hi)
        else:
            h = torch.zeros_like(h)
        return (h,) + rest if rest else h
    return hook


def hook_clipper(fault_type=FAULT_TYPE, fault_frac=FAULT_FRAC,
                  threshold=10.0):
    """
    Clipper — hard clips to [-threshold, +threshold].
    FT-ClipAct, Hoang et al. DATE 2020.
    Threshold tuned to typical BFloat16 activation range.
    """
    def hook(module, inp, out):
        h    = out[0] if isinstance(out, tuple) else out
        rest = out[1:] if isinstance(out, tuple) else None
        h    = _inject(h, fault_type, fault_frac)
        h    = torch.nan_to_num(h, nan=0.0,
                                 posinf=threshold, neginf=-threshold)
        h    = torch.clamp(h, -threshold, threshold)
        return (h,) + rest if rest else h
    return hook


def hook_fmapavg(fault_type=FAULT_TYPE, fault_frac=FAULT_FRAC):
    """
    FmapAvg — replaces corrupted activations with spatial/token mean.
    Ruospo et al. DATE 2023: 'Assessing CNN reliability'.
    Better than zero-fill because it uses local context.
    """
    def hook(module, inp, out):
        h    = out[0] if isinstance(out, tuple) else out
        rest = out[1:] if isinstance(out, tuple) else None
        h    = _inject(h, fault_type, fault_frac)
        bad  = torch.isnan(h) | torch.isinf(h)
        if bad.any():
            # Mean over token dimension (dim=1) — spatial average
            valid  = h.clone()
            valid[bad] = 0.0
            counts = (~bad).float()
            mean   = valid.sum(dim=1, keepdim=True) / \
                     (counts.sum(dim=1, keepdim=True) + 1e-8)
            h      = torch.where(bad, mean.expand_as(h), h)
        return (h,) + rest if rest else h
    return hook


def hook_selective_ranger(fault_type=FAULT_TYPE, fault_frac=FAULT_FRAC,
                           target_layers=None):
    """
    Selective Ranger — LFM2-specific contribution.
    Applies Ranger ONLY to LIV conv layers (0,1,3,5,7,9,11,13).
    GQA layers get zero-fill (cheaper, GQA equally sensitive).
    
    Rationale from your Experiment B:
      Both LIV and GQA show same mean drop.
      But LIV layers have multiplicative gates that can amplify
      out-of-range values more than GQA's additive attention.
      Therefore LIV layers need range-based recovery (Ranger)
      while GQA layers only need basic sanitization (zero-fill).
    
    This is your LFM2-specific method combining:
      Ranger (established, Wandel et al. 2021) +
      LFM2 architecture knowledge (novel application)
    """
    is_liv = target_layers is not None

    def hook(module, inp, out):
        h    = out[0] if isinstance(out, tuple) else out
        rest = out[1:] if isinstance(out, tuple) else None
        h    = _inject(h, fault_type, fault_frac)
        bad  = torch.isnan(h) | torch.isinf(h)

        if not bad.any():
            return (h,) + rest if rest else h

        if is_liv:
            # LIV layer: Ranger (range-based recovery)
            valid = h[~bad]
            if len(valid) > 0:
                mu  = valid.mean()
                sig = valid.std()
                lo  = mu - 6 * sig
                hi  = mu + 6 * sig
                h   = torch.nan_to_num(h, nan=mu.item(),
                                        posinf=hi.item(), neginf=lo.item())
                h   = torch.clamp(h, lo, hi)
            else:
                h = torch.zeros_like(h)
        else:
            # GQA layer: zero-fill (fast, sufficient)
            h = torch.nan_to_num(h, nan=0.0, posinf=0.0, neginf=0.0)

        return (h,) + rest if rest else h
    return hook


def hook_tmr_lite(fault_type=FAULT_TYPE, fault_frac=FAULT_FRAC):
    """
    TMR-lite — majority voting via median of three estimates.
    Triple Modular Redundancy adapted for activation faults.
    
    Standard TMR: run entire network 3x, vote on output.
    TMR-lite: for each corrupted activation, take median of
      [corrupted_value, 0, spatial_mean] — three estimates.
    This avoids 3x compute while retaining the voting principle.
    
    Novel: first application of TMR principle to LFM activation recovery.
    """
    def hook(module, inp, out):
        h    = out[0] if isinstance(out, tuple) else out
        rest = out[1:] if isinstance(out, tuple) else None
        h    = _inject(h, fault_type, fault_frac)
        bad  = torch.isnan(h) | torch.isinf(h)

        if bad.any():
            valid  = h.clone()
            valid[bad] = 0.0
            counts = (~bad).float()
            smean  = valid.sum(dim=1, keepdim=True) / \
                     (counts.sum(dim=1, keepdim=True) + 1e-8)
            smean  = smean.expand_as(h)

            # Three estimates: 0, spatial_mean, clean_neighbor
            # Median of [0, spatial_mean] for corrupted positions
            estimate = (smean * 0.5)   # average of 0 and mean
            h        = torch.where(bad, estimate, h)

        return (h,) + rest if rest else h
    return hook


# ══════════════════════════════════════════════════════════════════════════════
# EXPERIMENT
# ══════════════════════════════════════════════════════════════════════════════

def evaluate_with_hooks(model, tokenizer, hook_factories,
                         n_problems=100):
    """Register hooks, evaluate, remove hooks."""
    handles = []
    for li, hook_fn in hook_factories:
        h = model.model.layers[li].register_forward_hook(hook_fn)
        handles.append(h)

    lfm   = LFM2(model=model, tokenizer=tokenizer)
    bench = IFEval(n_problems=n_problems)
    bench.evaluate(model=lfm)
    score = bench.overall_score

    for h in handles:
        h.remove()
    return score


def run_comparison(model, tokenizer,
                   n_problems=100, n_seeds=5,
                   fault_type=FAULT_TYPE,
                   fault_frac=FAULT_FRAC):

    ALL_LAYERS = list(range(14))

    methods = {
        "1_clean": {
            "desc": "Clean baseline",
            "do_fault": False,
            "cite": "—",
        },
        "2_faulty": {
            "desc": "Faulty (no recovery)",
            "hook": hook_faulty,
            "cite": "fault model: Chai et al. 2025",
        },
        "3_zero_fill": {
            "desc": "Zero-fill",
            "hook": hook_zero_fill,
            "cite": "standard baseline",
        },
        "4_ranger": {
            "desc": "Ranger",
            "hook": hook_ranger,
            "cite": "Wandel et al. DATE 2021",
        },
        "5_clipper": {
            "desc": "Clipper",
            "hook": hook_clipper,
            "cite": "Hoang et al. DATE 2020",
        },
        "6_fmapavg": {
            "desc": "FmapAvg",
            "hook": hook_fmapavg,
            "cite": "Ruospo et al. DATE 2023",
        },
        "7_selective_ranger": {
            "desc": "Selective Ranger (LFM2)",
            "hook": None,   # special — different per layer type
            "cite": "novel: Ranger + LFM2 layer awareness",
        },
        "8_tmr_lite": {
            "desc": "TMR-lite",
            "hook": hook_tmr_lite,
            "cite": "TMR principle adapted for activations",
        },
    }

    all_scores = {k: [] for k in methods}

    for seed in range(n_seeds):
        print(f"\n── Seed {seed} ──────────────────────")
        random.seed(seed); torch.manual_seed(seed)

        for method_key, cfg in methods.items():

            # Clean — no hooks
            if not cfg.get("do_fault", True) and "hook" not in cfg:
                lfm   = LFM2(model=model, tokenizer=tokenizer)
                bench = IFEval(n_problems=n_problems)
                bench.evaluate(model=lfm)
                score = bench.overall_score

            # Selective Ranger — different hook per layer type
            elif method_key == "7_selective_ranger":
                hook_factories = [
                    (li, hook_selective_ranger(
                        fault_type, fault_frac,
                        target_layers=(li in LIV_LAYERS)
                    ))
                    for li in ALL_LAYERS
                ]
                score = evaluate_with_hooks(
                    model, tokenizer, hook_factories, n_problems
                )

            # All other methods — same hook on all layers
            else:
                hook_fn       = cfg["hook"]
                hook_factories = [
                    (li, hook_fn(fault_type, fault_frac))
                    for li in ALL_LAYERS
                ]
                score = evaluate_with_hooks(
                    model, tokenizer, hook_factories, n_problems
                )

            all_scores[method_key].append(score)
            print(f"  {cfg['desc']:<30}: {score:.4f}")

    # ── Paper table ────────────────────────────────────────────────────────────
    m_clean  = np.mean(all_scores["1_clean"])
    m_faulty = np.mean(all_scores["2_faulty"])
    drop     = m_clean - m_faulty

    print("\n" + "="*75)
    print("COMPARISON TABLE — Established Fault Recovery Methods on LFM2.5")
    print("="*75)
    print(f"Fault: {fault_type}  fraction={fault_frac}  "
          f"n_seeds={n_seeds}  n_problems={n_problems}")
    print(f"\n{'Method':<30} {'Score':<20} {'Drop':>8} "
          f"{'Recovery':>10}  Citation")
    print("-"*75)

    rows = []
    for method_key, cfg in methods.items():
        scores = all_scores[method_key]
        mean   = np.mean(scores)
        std    = np.std(scores)
        ci     = 1.96 * std / np.sqrt(len(scores)) if len(scores) > 1 else 0
        d      = m_clean - mean
        rec    = (mean - m_faulty) / (drop + 1e-8) \
                 if method_key not in ("1_clean", "2_faulty") else None
        rec_str = f"{rec:+.1%}" if rec is not None else "—"

        print(f"  {cfg['desc']:<28} {mean:.4f}±{ci:.4f}  "
              f"{d:>8.4f}  {rec_str:>10}  {cfg.get('cite','')}")

        rows.append({
            "method":    cfg["desc"],
            "mean":      round(mean, 4),
            "ci_95":     round(ci, 4),
            "drop":      round(d, 4),
            "recovery":  round(rec, 4) if rec is not None else None,
            "cite":      cfg.get("cite", ""),
        })

    df = pd.DataFrame(rows)
    df.to_csv("/kaggle/working/method_comparison.csv", index=False)
    print("\nSaved → /kaggle/working/method_comparison.csv")

    # Best method
    recovery_rows = [r for r in rows if r["recovery"] is not None]
    best = max(recovery_rows, key=lambda x: x["recovery"] or 0)
    print(f"\nBest recovery: {best['method']} ({best['recovery']:.1%})")
    print(f"Cite: {best['cite']}")

    return df


# Run
df = run_comparison(
    model, tokenizer,
    n_problems = 100,
    n_seeds    = 5,
    fault_type = "nan",
    fault_frac = 0.01,
)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/132 [00:00<?, ?it/s]


── Seed 0 ──────────────────────


README.md: 0.00B [00:00, ?B/s]

ifeval_input_data.jsonl: 0.00B [00:00, ?B/s]

...

Overall IFEval Accuracy: 0.6300
Instruction 'punctuation:no_comma' Accuracy: 0.4167
Instruction 'detectable_format:number_highlighted_sections' Accuracy: 0.1429
Instruction 'length_constraints:number_words' Accuracy: 0.2500
Instruction 'detectable_content:number_placeholders' Accuracy: 0.0000
Instruction 'combination:repeat_prompt' Accuracy: 1.0000
Instruction 'detectable_format:title' Accuracy: 1.0000
Instruction 'change_case:english_lowercase' Accuracy: 0.0000
Instruction 'detectable_format:number_bullet_lists' Accuracy: 1.0000
Instruction 'change_case:english_capital' Accuracy: 1.0000
Instruction 'detectable_format:multiple_sections' Accuracy: 1.0000
Instruction 'change_case:capital_word_frequency' Accuracy: 1.0000
Instruction 'startend:quotation' Accuracy: 1.0000
Instruction 'keywords:existence' Accuracy: 1.0000
Instruction 'detectable_format:json_format' Accuracy: 1.0000
Instruction 'length_constraints:number_paragraphs' Accuracy: 1.0000
Instruction 'combination:two_responses' Acc

Processing 100 IFEval problems: 100%|██████████| 100/100 [00:56<00:00,  1.78it/s]


Overall IFEval Accuracy: 0.6200
Instruction 'punctuation:no_comma' Accuracy: 0.4167
Instruction 'detectable_format:number_highlighted_sections' Accuracy: 0.2857
Instruction 'length_constraints:number_words' Accuracy: 0.2500
Instruction 'detectable_content:number_placeholders' Accuracy: 0.0000
Instruction 'combination:repeat_prompt' Accuracy: 1.0000
Instruction 'detectable_format:title' Accuracy: 1.0000
Instruction 'change_case:english_lowercase' Accuracy: 0.0000
Instruction 'detectable_format:number_bullet_lists' Accuracy: 1.0000
Instruction 'change_case:english_capital' Accuracy: 1.0000
Instruction 'detectable_format:multiple_sections' Accuracy: 1.0000
Instruction 'change_case:capital_word_frequency' Accuracy: 1.0000
Instruction 'startend:quotation' Accuracy: 1.0000
Instruction 'keywords:existence' Accuracy: 1.0000
Instruction 'detectable_format:json_format' Accuracy: 1.0000
Instruction 'length_constraints:number_paragraphs' Accuracy: 1.0000
Instruction 'combination:two_responses' Acc

Processing 100 IFEval problems: 100%|██████████| 100/100 [00:00<00:00, 173.58it/s]

Overall IFEval Accuracy: 0.5000
Instruction 'punctuation:no_comma' Accuracy: 1.0000
Instruction 'detectable_format:number_highlighted_sections' Accuracy: 0.0000
Instruction 'length_constraints:number_words' Accuracy: 0.2500
Instruction 'detectable_content:number_placeholders' Accuracy: 0.0000
Instruction 'combination:repeat_prompt' Accuracy: 0.0000
Instruction 'detectable_format:title' Accuracy: 0.0000
Instruction 'change_case:english_lowercase' Accuracy: 0.0000
Instruction 'detectable_format:number_bullet_lists' Accuracy: 1.0000
Instruction 'change_case:english_capital' Accuracy: 1.0000
Instruction 'detectable_format:multiple_sections' Accuracy: 1.0000
Instruction 'change_case:capital_word_frequency' Accuracy: 1.0000
Instruction 'startend:quotation' Accuracy: 1.0000
Instruction 'keywords:existence' Accuracy: 1.0000
Instruction 'detectable_format:json_format' Accuracy: 1.0000
Instruction 'length_constraints:number_paragraphs' Accuracy: 1.0000
Instruction 'combination:two_responses' Acc